# 08 · SC-FEGAN 통합 — 자동 입력 생성 + Inference

**Phase 7-A 마지막 산출물**: project/input_generator의 sketch/color/mask/composer로 SC-FEGAN 9채널 입력을 자동 생성하고, 시술 3종(코끝/쌍커풀/V라인) Before/After를 시각화한다.

**환경**: Colab (T4 GPU) — TF 1.x compat 모드 필요.

**산출물**: `outputs/integration_<procedure>.png` 6장 (시술 3종 × Before/After)

---

## 1. Colab 환경 셋업

- cv-final repo clone
- mediapipe 0.10.21 (이전 호환 버전)
- TF 2.x + tf-keras + legacy keras 환경변수

In [ ]:
import os, sys
IS_COLAB = "google.colab" in sys.modules
print(f"Colab: {IS_COLAB}")

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO = '/content/cv-final'
    if not os.path.exists(REPO):
        !git clone https://github.com/kim-cv-final/cv-final.git $REPO
    %cd $REPO
else:
    REPO = os.path.abspath('..')
    os.chdir(REPO)

print(f'CWD: {os.getcwd()}')

In [ ]:
if IS_COLAB:
    !pip install -q mediapipe==0.10.21 opencv-python pyyaml
    os.environ['TF_USE_LEGACY_KERAS'] = '1'
    print('✅ 의존성 설치 완료')

## 2. Imports + 헬퍼

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

from project.input_generator import (
    compose_scfegan_input, to_scfegan_tensor, load_procedures,
)

OUTPUT_DIR = Path('outputs/integration')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SIZE = 512
PROCEDURES = load_procedures()
print(f'Procedures: {list(PROCEDURES.keys())}')

## 3. 입력 이미지 + MediaPipe 478 랜드마크 추출

In [ ]:
# 데모 이미지 경로 (Colab에서 직접 업로드하거나 Drive 경로 사용)
if IS_COLAB:
    from google.colab import files
    print('얼굴 사진 1장 업로드:')
    uploaded = files.upload()
    IMG_PATH = list(uploaded.keys())[0]
else:
    # 로컬 테스트용 더미 이미지
    IMG_PATH = 'demo_face.jpg'
    if not Path(IMG_PATH).exists():
        dummy = np.full((SIZE, SIZE, 3), [220, 195, 175], dtype=np.uint8)
        cv2.imwrite(IMG_PATH, cv2.cvtColor(dummy, cv2.COLOR_RGB2BGR))

image_bgr = cv2.imread(IMG_PATH)
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
image_rgb = cv2.resize(image_rgb, (SIZE, SIZE))

plt.figure(figsize=(5, 5))
plt.imshow(image_rgb); plt.axis('off'); plt.title('Input')
plt.show()

In [ ]:
def extract_landmarks(image_rgb: np.ndarray, size: int = 512) -> np.ndarray:
    """MediaPipe Face Mesh로 478 랜드마크 (x, y) 추출."""
    import mediapipe as mp
    mp_face = mp.solutions.face_mesh
    with mp_face.FaceMesh(
        static_image_mode=True, max_num_faces=1, refine_landmarks=True,
    ) as fm:
        res = fm.process(image_rgb)
    if not res.multi_face_landmarks:
        raise RuntimeError('얼굴 감지 실패 — 다른 사진 사용')
    lms = res.multi_face_landmarks[0].landmark
    h, w = image_rgb.shape[:2]
    pts = np.array([[lm.x * w, lm.y * h] for lm in lms], dtype=np.float32)
    pts = (pts * (size / w)).astype(np.int32) if w != size else pts.astype(np.int32)
    return pts

landmarks = extract_landmarks(image_rgb, size=SIZE)
print(f'Landmarks: {landmarks.shape}')

# 시각화
vis = image_rgb.copy()
for x, y in landmarks:
    cv2.circle(vis, (int(x), int(y)), 1, (0, 255, 0), -1)
plt.figure(figsize=(5, 5))
plt.imshow(vis); plt.axis('off'); plt.title('478 Landmarks')
plt.show()

## 4. 시술 3종 × 9채널 입력 자동 생성

In [ ]:
def visualize_composed(composed: dict, title: str = ''):
    """compose_scfegan_input 결과 dict의 5채널을 한 화면에 시각화."""
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    axes[0].imshow(composed['incomplete_image']); axes[0].set_title('Incomplete')
    axes[1].imshow(composed['sketch'][..., 0], cmap='gray'); axes[1].set_title('Sketch')
    axes[2].imshow(composed['color']); axes[2].set_title('Color')
    axes[3].imshow(composed['mask'][..., 0], cmap='gray'); axes[3].set_title('Mask')
    axes[4].imshow(composed['noise'][..., 0], cmap='gray'); axes[4].set_title('Noise')
    for ax in axes:
        ax.axis('off')
    fig.suptitle(title)
    plt.tight_layout(); plt.show()
    return fig

composed_all = {}
for proc_id in ['nose_tip', 'double_eyelid', 'jaw_v_line']:
    composed = compose_scfegan_input(
        image_rgb, landmarks, proc_id, size=SIZE,
        intensity=0.6, seed=42,
    )
    composed_all[proc_id] = composed
    visualize_composed(composed, title=proc_id)

## 5. SC-FEGAN Inference (TF 1.x compat)

**주의**: SC-FEGAN 원 코드는 TF 1.x 기반. Colab Python 3.11 + TF 2.x compat 모드 사용.

ckpt가 없거나 환경 실패 시 입력 시각화만 진행.

In [ ]:
SCFEGAN_CKPT = '/content/drive/MyDrive/SC-FEGAN-ckpt/SC-FEGAN.ckpt'
scfegan_available = Path(SCFEGAN_CKPT + '.index').exists() if IS_COLAB else False
print(f'SC-FEGAN ckpt available: {scfegan_available}')

if scfegan_available:
    import tensorflow.compat.v1 as tf
    tf.disable_v2_behavior()
    # 실제 SC-FEGAN 모델 로드 + inference
    # (project/scfegan_finetune/inference.py — Phase 7-C1에서 작성 예정)
    print('TF 1.x compat 모드 활성화')
else:
    print('⚠ ckpt 미설치 — 입력 시각화만 진행. Phase 7-C1에서 ckpt 셋업 예정.')

In [ ]:
def run_scfegan_inference(composed: dict) -> np.ndarray:
    """SC-FEGAN ckpt → (size, size, 3) RGB inpainted 이미지.

    Phase 7-C1 fine-tuned ckpt도 호환 (동일 graph).
    실패 시 incomplete_image를 그대로 반환 (폴백).
    """
    if not scfegan_available:
        return composed['incomplete_image']
    try:
        tensor = to_scfegan_tensor(
            composed, normalize=True, channels_first=False, batch_dim=True,
        )
        # TODO: 실제 graph forward (Phase 7-C1에서 import)
        # output = sess.run(generator_output, feed_dict={input_ph: tensor})
        # return ((output[0] + 1) * 127.5).clip(0, 255).astype(np.uint8)
        return composed['incomplete_image']  # placeholder
    except Exception as e:
        print(f'SC-FEGAN inference 실패 → 폴백: {e}')
        return composed['incomplete_image']

results = {}
for proc_id, composed in composed_all.items():
    results[proc_id] = run_scfegan_inference(composed)
print('Inference 완료:', list(results.keys()))

## 6. Before / After 시각화 (시술 3종)

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(10, 14))
for i, (proc_id, after) in enumerate(results.items()):
    axes[i, 0].imshow(image_rgb); axes[i, 0].set_title(f'{proc_id} · Before')
    axes[i, 1].imshow(after);     axes[i, 1].set_title(f'{proc_id} · After')
    for ax in axes[i]:
        ax.axis('off')
plt.tight_layout()

save_path = OUTPUT_DIR / 'before_after_3procs.png'
plt.savefig(save_path, dpi=120, bbox_inches='tight')
print(f'✅ 저장: {save_path}')
plt.show()

In [ ]:
# 시술별 개별 저장
for proc_id, after in results.items():
    out_path = OUTPUT_DIR / f'integration_{proc_id}.png'
    cv2.imwrite(str(out_path), cv2.cvtColor(after, cv2.COLOR_RGB2BGR))
    print(f'✅ {out_path}')
print('\nPhase 7-A 통합 검증 완료.')

---
## 다음 단계 (Phase 7-B/C1)

- **7-B**: Refinement Network 학습 — SC-FEGAN 출력의 artifact 제거
- **7-C1**: SC-FEGAN G+D fine-tuning — 본 노트북의 `run_scfegan_inference` placeholder 교체

본 노트북은 두 단계 완료 후 재실행하면 진짜 inference 결과를 볼 수 있다.